In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"01a0c36b-d704-744a-8ee1-89ec24ae6741\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Pave Bank - Fintech Loan Analysis\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

In [ ]:
# import jinja2
# raw_query = """
#     SELECT COUNT(*) AS loans FROM `pave-bank-fintech.fintech_test.v_loans`   -- To check if HEX is connected to data in BigQuery
# """
# sql_query = jinja2.Template(raw_query).render(vars())

# Fintech loan analysis — read this first: data limitations and how I adapted

**Dataset:** 270,299 loans originated 2012–2019 (latest issue month Dec 2019), one loan per customer, loaded to BigQuery and analysed with SQL and Python. Raw tables were loaded unchanged with explicit schemas and checked against the source files (row counts, sums, NULL counts). All cleaning and definitions live in one view, `v_loans`, which every task reads.

## 1. The brief describes data this dataset does not contain

| The brief assumed | The dataset actually has | What I did instead |
|---|---|---|
| Separate tables such as loans, **payments**, customers, transactions | Six files: `customer`, `loan`, `loan_with_region`, `loan_count_by_year`, `loan_purposes`, `state_region`. **No payments or transactions table** | The only join available is loan to customer (matches 1:1, 270,299 rows each) |
| Statuses *active*, *paid_off*, *defaulted* | Seven statuses (Current, Fully Paid, Charged Off, Default, In Grace Period, Late 16–30 days, Late 31–120 days) | **paid_off** = Fully Paid; **defaulted** = Charged Off + Default; **active** = the four other statuses |
| Task 3: join loans and payments to find customers with late payments | No payment dates, no payment history, no days-late column | **"Late" = current status is In Grace Period, Late 16–30 or Late 31–120.** This is a snapshot, not a payment history: 5,614 of 176,075 active loans (3.19%). Because each customer has one loan, "frequency" became the *share of active loans that are late* within an interest-rate band, not a count per customer |
| Task 4: group customers by average delay or credit utilisation | Neither exists (no payment history, no credit limits) | Grouped on loan amount, interest rate, term, income, loan-to-income, and **balance ÷ income as a leverage proxy** (total current balance includes mortgages, so it is *not* credit utilisation) |

## 2. Data problems found, and how they were handled
- **Loan statuses appear to come from different snapshot dates.** 86% of 2016 loans are still "active" (for both 36- and 60-month terms) although the data runs to Dec 2019, so year-to-year default counts are not fully comparable. A plain default rate is biased low for recent years (open loans cannot default yet), so Task 2 shows it two ways: against all loans and against resolved loans only. Cause unproven.
- **Only approved loans are present,** and there is no outcome date. We cannot see rejected applicants or when a loan defaulted.
- **Hashed customer IDs** stored as unreadable text: used only as a join key; `loan_id` is the readable handle.
- **Messy fields** cleaned in the view (raw tables untouched): five spellings of application type reduced to three; a leading space in `term`; `n/a` used as a missing-value code (18,745 rows); a stray header row inside `state_region` (51 real states); duplicate/constant columns (`issue_date`, `notes`) dropped.
- **Extreme values:** income from $34 to $9.55M and loan-to-income up to 175. Capped at the 1st/99th percentile for clustering and modelling; no customer was deleted.
- **Joint income is missing for 93% of customers** by design (only joint applications have it), so only the primary applicant's income is used.

# **TASK 1**

In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#       issue_year,
#       status_group,                                -- paid_off --> fully paid, defaulted --> charged off or defaulted, active --> current, grace period, late
#       COUNT(*) AS loans                            -- amount of loans for the period
#     FROM `pave-bank-fintech.fintech_test.v_loans`
#     GROUP BY issue_year, status_group
#     ORDER BY issue_year, status_group;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     -- Share of loans still active, by origination year and loan term (36 vs 60 months).
#     -- One row per year. "Active" = status_group 'active' in v_loans (Current, In Grace Period,
#     -- Late 16-30, Late 31-120); the rest are paid_off or defaulted.
#     -- Each share = active loans of that term / ALL loans of that term, within the year.
#     
#     SELECT
#       issue_year AS year,                                        -- the group key: one output row per year
#       -- Numerator: 36-month loans still active. Denominator: all 36-month loans in the year.
#       -- COUNTIF counts only the rows where its condition is true, so the two terms can sit side by side.
#       FORMAT('%.1f%%', 100 * COUNTIF(term_months = 36 AND status_group = 'active') / COUNTIF(term_months = 36))
#         AS share_active_36_month,                                -- text such as '86.1%'
#       -- Same calculation for 60-month loans.
#       FORMAT('%.1f%%', 100 * COUNTIF(term_months = 60 AND status_group = 'active') / COUNTIF(term_months = 60))
#         AS share_active_60_month
#     FROM `pave-bank-fintech.fintech_test.v_loans`                -- the cleaned view: one row per loan
#     GROUP BY issue_year                                          -- COUNTIF runs once per year
#     ORDER BY issue_year;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

When we counted the outcomes for each origination year, 2016 stood out: 86.1% of 2016 loans are still "active", and this is the same for 36-month loans (86.1%) and 60-month loans (86.1%). The latest loan in the data was issued in December 2019, so a 36-month loan issued in 2016 should have finished by the end of 2019 at the latest. Yet 86% of them are recorded as still open. 2014 is also unusual: 29% of its 36-month loans are still open, while for 2013 and 2015 the figure is 0%.



As for my working explanation (a hypothesis, not proven) is that the statuses were not all recorded on the same date: some years were probably recorded at one point in time and some at another. This matters because it makes year-to-year comparisons of default counts unreliable, which is why Task 2 shows the default rate in two ways.



Defined the groups for the chart below as: 

paid off means --> Fully Paid

defaulted means --> Charged Off or Default

active means --> Current or late-stage but still open



### What chart shows:
1. Origination volume grew from 2,594 loans in 2012 to 51,737 in 2019.

In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#       issue_year,
#       status_group,
#       COUNT(*) AS loans,
#       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY issue_year), 2) AS pct_of_year     -- each status share (%) of its year's total loans, rounded to 2 decimal places
#     FROM `pave-bank-fintech.fintech_test.v_loans`
#     GROUP BY issue_year, status_group
#     ORDER BY issue_year, status_group;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

### What the charts show:

1. For 2012–2015 originations, **14.5–18.2%** of loans ended in default.

2. 2016 stands out: **86%** of 2016 loans are still "active", while only **2.6%** have defaulted. This is consistent across both 36- and 60-month terms, with **86.1%** still active in each case, so loan term does not appear to explain the difference.

3. 2018–2019 loans are **over 90%** active, which is expected given that these are more recent originations.

### Interesting findings:

1. The latest loan in the data was issued in December 2019, so a 36-month loan issued in 2016 should already have matured. However, **86%** of these loans are still marked as "active".

2. This suggests that loan statuses may not all be recorded as of the same date. As a result, year-to-year default counts should be interpreted with caution and compared carefully.

3. Task 2 addresses this by also measuring the default rate among resolved loans only.

# **TASK 2**

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 2: loan performance by origination year.
#     -- Returns one row per year with the number of loans, average interest rate,
#     -- average principal, and the default rate measured two ways
#     -- (against all loans, and against resolved loans only).
#     
#     SELECT
#       issue_year,                                                                                  -- origination year of the loan
#       COUNT(*)                                                    AS loans,                        -- number of loans originated that year
#       ROUND(AVG(int_rate) * 100, 2)                               AS avg_int_rate_pct,             -- average interest rate, converted from a fraction to a percentage
#       ROUND(AVG(loan_amount), 2)                                  AS avg_principal,                -- average loan amount (principal) in dollars
#       COUNTIF(is_default)                                         AS defaulted,                    -- number of loans that are Charged Off or in Default
#       COUNTIF(is_resolved)                                        AS resolved,                     -- number of loans with a known outcome (paid off or defaulted)
#       ROUND(100 * COUNTIF(is_default) / COUNT(*), 2)              AS default_rate_all_pct,         -- % of ALL loans that defaulted (looks too low for recent years: open loans can't default yet)
#       ROUND(100 * COUNTIF(is_default) / COUNTIF(is_resolved), 2)  AS default_rate_resolved_pct     -- % of RESOLVED loans that defaulted (ignores loans still open)
#     FROM `pave-bank-fintech.fintech_test.v_loans`  -- cleaned view, one row per loan
#     GROUP BY issue_year                            -- one result row per origination year
#     ORDER BY issue_year;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 2 (chart data): default rate by origination year, measured two ways.
#     -- The two rates are stacked on top of each other (UNION ALL) into "long" format:
#     -- one row per year per measure, which lets Hex draw one line per measure.
#     
#     -- Part 1: default rate against ALL loans
#     SELECT issue_year,                                                                      -- origination year
#            'All loans' AS measure,                                                          -- label for this line on the chart
#            ROUND(100 * COUNTIF(is_default) / COUNT(*), 2) AS default_rate_pct               -- % of all loans that defaulted
#     FROM `pave-bank-fintech.fintech_test.v_loans`                                           -- cleaned view, one row per loan
#     GROUP BY issue_year                                                                     -- one row per year
#     
#     UNION ALL                                                                               -- append part 2's rows below part 1's (keeps every row)
#     
#     -- Part 2: default rate against RESOLVED loans only
#     SELECT issue_year,                                                                      -- origination year
#            'Resolved loans only' AS measure,                                                -- label for this line on the chart
#            ROUND(100 * COUNTIF(is_default) / COUNTIF(is_resolved), 2) AS default_rate_pct   -- % of paid-off + defaulted loans that defaulted
#     FROM `pave-bank-fintech.fintech_test.v_loans`
#     GROUP BY issue_year
#     
#     ORDER BY issue_year, measure;                                                           -- sort the combined result by year, then by measure
# """
# sql_query = jinja2.Template(raw_query).render(vars())

### Averages are stable

The average interest rate remained between 12.6% and 14.5%, while the average principal stayed roughly between $13.7k and $16.5k across all origination years from 2012 to 2019. Neither metric appears to explain the swings in default rates discussed below.

### Two definitions of default rate

- **All loans:** defaulted loans / all loans originated in that year
- **Resolved loans only:** defaulted loans / (paid off + defaulted loans), excluding loans that are still open

### What the chart shows:

1. **The two definitions tell different stories from 2016 onward.** On an all-loans basis, the default rate drops sharply from 18.2% in 2015 to 2.6% in 2016. On a resolved-only basis, however, it changes only slightly, from 20.2% to 18.6%.

2. **The gap between the two measures increases from 0.6 percentage points in 2012 to 16.0 percentage points in 2016.** This gap reflects the share of each cohort that is still unresolved, so it is more a measure of unfinished loans than credit quality.

3. **For 2012–2017, the resolved-only default rate remains within a 15.6%–22.4% range**, meaning roughly one in five resolved loans ended in default. The highest rate was 22.4% for loans originated in 2014.

4. **The lower rates in 2018 (8.7%) and 2019 (13.4%) should not be interpreted as improved credit quality.** These figures are based on relatively few resolved loans — 3,659 and 4,546 respectively, compared with 37,711 for 2015. Loans that resolve early are unlikely to be representative of the full cohort.

### Takeaway

The all-loans default rate combines loan performance with loan age, making comparisons across origination years difficult. The resolved-only rate provides a more consistent basis for comparing the 2012–2017 cohorts. However, neither measure is completely clean because the loan statuses appear not to come from a single snapshot date, as noted in the Task 1 caveat.

### Interesting findings:

1. **2014 also stands out as unusual.** It has the highest resolved-only default rate at 22.4%, while 29% of its 36-month loans are still marked as active, even though the data runs through December 2019. This compares with 0% for both 2013 and 2015. The chart alone does not provide enough information to explain this difference, so it is best to state that plainly if asked.

2. **Looking only at the grey line could give a misleading impression.** It might suggest that lending quality improved roughly seven-fold in 2016, but the resolved-only rate shows that this apparent improvement is largely driven by the way unresolved loans are counted rather than by a comparable improvement in loan performance.

# **TASK 3**

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 3, step 1: what "late" looks like among ACTIVE loans.
#     -- Counts loans by their exact status, marks which statuses count as late (is_late),
#     -- and shows each status's share of all active loans.
#     
#     SELECT
#       loan_status,                                                        -- exact status text (Current, In Grace Period, Late (16-30 days) ...)
#       is_late,                                                            -- true if the status counts as late (our Task 3 definition)
#       COUNT(*) AS loans,                                                  -- number of active loans with this status
#       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_active   -- this status's % of all active loans (OVER () = the whole result is one group)
#     FROM `pave-bank-fintech.fintech_test.v_loans`                         -- cleaned view, one row per loan
#     WHERE status_group = 'active'                                         -- keep only open loans (Current + the three late statuses); only these can be late
#     GROUP BY loan_status, is_late                                         -- one row per status
#     ORDER BY loans DESC;                                                  -- largest group first
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 3, step 2: the customers with late payments.
#     -- One row per late loan (one customer, since each customer has one loan),
#     -- with the loan's and borrower's main characteristics.
#     -- Sorted most severe first, then largest loan first.
#     
#     SELECT
#       loan_id,                                      -- readable ID for the loan/customer (customer_id itself is an unreadable hash)
#       loan_status,                                  -- how late: Late (31-120 days), Late (16-30 days) or In Grace Period
#       loan_amount,                                  -- loan size in dollars
#       ROUND(int_rate * 100, 2) AS int_rate_pct,     -- interest rate converted from a fraction to a percentage (0.0799 -> 7.99)
#       grade,                                        -- the lender's risk grade, A (best) to G (worst)
#       term_months,                                  -- loan length: 36 or 60 months
#       issue_year,                                   -- year the loan was originated
#       application_type,                             -- Individual, Joint or Direct Pay
#       home_ownership,                               -- MORTGAGE, RENT, OWN, ...
#       annual_inc,                                   -- borrower's annual income
#       ROUND(loan_to_income, 3) AS loan_to_income,   -- loan amount divided by annual income, rounded to 3 decimals
#       state                                         -- US state
#     FROM `pave-bank-fintech.fintech_test.v_loans`   -- cleaned view, one row per loan
#     WHERE is_late                                   -- keep only late loans (true/false column, so no "= true" needed)
#     ORDER BY
#       CASE loan_status                              -- custom sort order: severity, not alphabetical
#         WHEN 'Late (31-120 days)' THEN 1            -- most severe first
#         WHEN 'Late (16-30 days)'  THEN 2
#         ELSE 3                                      -- everything else late = In Grace Period, last
#       END,
#       loan_amount DESC,                             -- within each severity, largest loans first
#       loan_id;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 3, step 3: late-payment rate by interest-rate band and loan term.
#     -- Loans are grouped into 2-point interest-rate bands (4-6%, 6-8%, ...) and by term (36/60 months);
#     -- for each group it counts active loans, how many are late, and the late rate.
#     -- Feeds the Task 3 bubble chart.
#     
#     -- Step A: keep only active loans and compute each loan's rate band
#     WITH active AS (
#       SELECT
#         -- Rate band: convert the rate to a percentage (rounded to 2 decimals to avoid floating-point noise),
#         -- divide by 2, round DOWN to a whole number, multiply by 2 -> lower edge of a 2-point band.
#         -- Example: 0.1349 -> 13.49 -> 6.745 -> 6 -> 12, so this loan is in the "12-14%" band.
#         CAST(FLOOR(ROUND(int_rate * 100, 2) / 2) * 2 AS INT64) AS band_start_pct,
#         term_months,                                              -- 36 or 60
#         is_late                                                   -- true if the loan counts as late
#       FROM `pave-bank-fintech.fintech_test.v_loans`               -- cleaned view, one row per loan
#       WHERE status_group = 'active'                               -- only open loans can be late, so they are the denominator
#     )
#     
#     -- Step B: count and compare within each band and term
#     SELECT
#       band_start_pct,                                                                                     -- lower edge of the band (4, 6, 8, ... 30)
#       CONCAT(CAST(band_start_pct AS STRING), '-', CAST(band_start_pct + 2 AS STRING), '%') AS rate_band,  -- readable label, e.g. "12-14%"
#       term_months,                                                                                        -- loan term
#       COUNT(*)                                        AS active_loans,                                    -- active loans in this band and term
#       COUNTIF(is_late)                                AS late_loans,                                      -- how many of them are late
#       ROUND(100 * COUNTIF(is_late) / COUNT(*), 2)     AS late_rate_pct                                    -- late loans as a % of active loans in the group
#     FROM active                                                                                           -- read from the step A result
#     GROUP BY band_start_pct, term_months                                                                  -- one row per band and term
#     ORDER BY band_start_pct, term_months;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# Task 3 bubble chart: late-payment rate vs interest rate, split by loan term.
# Each bubble is one interest-rate band for one term; bubble size = number of active loans behind it.
# Input: task3_late_by_rate, the result of the Hex SQL cell (28 rows: 14 rate bands x 2 terms).

import plotly.express as px   # Plotly Express: high-level library for interactive charts

df = task3_late_by_rate.copy()                                                                     # work on a copy so the original SQL result stays unchanged
df["term"] = df["term_months"].astype(int).astype(str) + " months"                                 # 36 -> "36 months" (text, so it is treated as a category/colour, not a number)
df["band_mid_pct"] = df["band_start_pct"] + 1                                                      # centre of each 2-point band (the 4-6% band is plotted at 5)

fig = px.scatter(
    df,                                                                                            # the data
    x="band_mid_pct",                                                                              # X axis: interest-rate band midpoint
    y="late_rate_pct",                                                                             # Y axis: % of active loans that are late
    size="active_loans",                                                                           # bubble size: number of active loans in the group (bigger = more loans)
    color="term",                                                                                  # one colour per loan term
    color_discrete_map={"36 months": "#7b5cd6", "60 months": "#f28e2b"},                           # fixed colours: purple for 36, orange for 60
    category_orders={"term": ["36 months", "60 months"]},                                          # legend order
    size_max=50,                          # largest bubble is 50 pixels wide
    hover_name="rate_band",               # bold title in the hover box, e.g. "12-14%"
    hover_data={"active_loans": ":,", "late_loans": ":,", "late_rate_pct": ":.2f",                 # extra hover fields with formatting (thousands separators, 2 decimals)
                "band_mid_pct": False, "term": False},                                             # False = hide these fields from the hover box
    labels={"late_rate_pct": "Late rate (%)", "active_loans": "Active loans",                      # friendly names shown in the hover box and legend
            "late_loans": "Late loans", "term": "Loan term"},
    title="Late-payment rate vs interest rate (active loans; bubble size = number of loans)",
)

# Bubble styling: smallest bubble is 4 px so tiny groups stay visible; slightly see-through; thin white outline
fig.update_traces(marker=dict(sizemin=4, opacity=0.75, line=dict(width=1, color="white")))

# X axis: title, and tick marks every 2 points starting at 5 (5, 7, 9, ...) so each tick sits on a band midpoint
fig.update_xaxes(title="Interest rate band midpoint (%)", tick0=5, dtick=2)

# Y axis: title, and always start at 0 so bar heights are not visually exaggerated
fig.update_yaxes(title="Late rate (% of active loans)", rangemode="tozero")

fig   # last line of the cell: Hex displays the chart

# Task 3 — Payment behaviour

### How "late" is defined

The dataset has no payment history, so a loan is considered late if its current status is **In Grace Period**, **Late (16–30 days)**, or **Late (31–120 days)**. This is a snapshot of current status, not a count of missed payments.

Only open loans can be late, so late rates are calculated among active loans, including **Current** and the three late-status categories. Overall, **5,614 of 176,075 active loans (3.19%)** are currently late. Each customer has exactly one loan, so these represent 5,614 customers, which are listed in the table above.

### What the chart shows

1. **Late rates rise steadily with interest rates.** Across all active loans, the rate increases from 0.6% in the 4–6% band to 14.9% in the 30–32% band, making the highest rate roughly 23 times higher.

2. **The pattern also holds within each loan term.** For 36-month loans, the late rate increases from 0.65% to 17.65%, while for 60-month loans it rises from 0.68% in the 6–8% band to 14.25%. This suggests the relationship is not simply caused by mixing the two loan terms.

3. **36-month loans are late more often than 60-month loans at the same interest rate in 11 of the 14 bands.** We did not investigate the reason for this difference.

4. **The highest-rate groups are relatively small.** The 30–32% band contains 558 active loans, while the 28–30% dip for 36-month loans, at 4.95%, is based on 303 loans, only 15 of which are late. Individual points in these smaller groups should therefore be interpreted with some caution.

### Interpretation

Interest rates are set by the lender based on assessed risk, so the pattern in the chart is consistent with risk-based pricing. It does not show that higher interest rates cause loans to become late.

### Where each claim comes from

1. The 0.6% and 14.9% figures come from the band-level totals: 4,053 active loans with 26 late, and 558 active loans with 83 late.

2. The 36-month and 60-month ranges, along with the "11 of 14 bands" comparison, come from the 28-row table validated in Step 3.

3. The final interpretation is an analytical observation rather than a direct measurement from the dataset.

# **TASK 4**

In [ ]:
# import jinja2
# raw_query = """
#     -- Task 4: customer-level table for segmentation (the clustering itself happens later in a Python cell).
#     -- One row per loan = one row per customer (each customer has exactly one loan).
#     -- Pulls the columns used to build the customer groups, plus columns used only to describe the groups afterwards.
#     
#     SELECT
#       loan_id,                                                      -- readable ID for each customer (customer_id itself is an unreadable hash)
#     
#       -- Clustering inputs (used to build the groups):
#       loan_amount,                                                  -- loan size in dollars
#       ROUND(int_rate * 100, 2) AS int_rate_pct,                     -- interest rate converted from a fraction to a percentage
#       term_months,                                                  -- loan length: 36 or 60 months
#       annual_inc,                                                   -- borrower's annual income
#       loan_to_income,                                               -- loan amount divided by annual income
#       tot_cur_bal,                                                  -- total current balance across the borrower's accounts (not a clustering input on its own; used for the ratio below)
#       SAFE_DIVIDE(tot_cur_bal, annual_inc) AS balance_to_income,    -- total balance divided by income: a leverage proxy (returns NULL instead of an error if income is 0)
#     
#       -- Description only (NOT used to build the groups):
#       grade,                                                        -- lender's risk grade, A to G
#       application_type,                                             -- Individual, Joint or Direct Pay
#       home_ownership,                                               -- MORTGAGE, RENT, OWN, ...
#       issue_year,                                                   -- year the loan was originated
#       status_group,                                                 -- paid_off, defaulted or active
#       is_default,                                                   -- true if the loan defaulted
#       is_late                                                       -- true if the loan counts as late
#     FROM `pave-bank-fintech.fintech_test.v_loans`;                  -- cleaned view, one row per loan; no WHERE, so all 270,299 loans are returned
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
!pip --python /ipython/.venv/bin/python install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 145.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 252.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [ ]:
# Task 4, step 2: prepare the six clustering features.
# Clustering (k-means) measures distances between customers, so features must be on comparable scales
# and free of extreme outliers. Three steps: (1) cap extreme values, (2) log-transform skewed features,
# (3) standardise every feature to mean 0 and standard deviation 1. No customers are removed.
# Output: X_scaled (used for clustering) and a summary table, report.

import numpy as np                                              # numerical functions (log, etc.)
import pandas as pd                                             # tables (DataFrames)
from sklearn.preprocessing import StandardScaler                # rescales each column to mean 0, std 1

df = task4_customers.copy()                                     # SQL result from Hex; a copy keeps the original unchanged
# the six features the clusters are built from
FEATURES = ["loan_amount", "int_rate_pct", "term_months", "annual_inc", "loan_to_income", "balance_to_income"]
# the three features with a long right tail (get a log transform)
SKEWED = ["annual_inc", "loan_to_income", "balance_to_income"]

print("rows:", len(df), "| columns:", df.shape[1])              # number of customers and columns
# total missing cells across the six features
print("NULLs in clustering features:", int(df[FEATURES].isna().sum().sum()))

raw = df[FEATURES].astype(float)                                # the six feature columns, forced to plain decimals

# 1. cap extreme values at the 1st / 99th percentile (rows are kept)
lo, hi = raw.quantile(0.01), raw.quantile(0.99)                 # per column: the 1st and 99th percentile values
capped = raw.clip(lower=lo, upper=hi, axis=1)                   # pull values below lo up to lo, above hi down to hi
n_capped = ((raw < lo) | (raw > hi)).sum()                      # per column: how many values were capped (True = 1)

# 2. log-transform the heavily skewed features
X = capped.copy()                                               # X = capped features; the skewed ones are logged below
for c in SKEWED:
    X[c] = np.log1p(X[c])                                       # log(1 + x): compresses huge values; the +1 keeps zeros valid

# 3. standardise: mean 0, standard deviation 1
# (value - column mean) / column std, for every feature; keep names and row order
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=FEATURES, index=df.index)

report = pd.DataFrame({                                         # summary table to check the preparation worked
    # cut-off values used, and how many customers were capped, per feature
    "cap_low_1%": lo, "cap_high_99%": hi, "values_capped": n_capped,
    "skew_raw": raw.skew(), "skew_prepared": X.skew(),          # lopsidedness before and after (0 = symmetric)
    # should be about 0 and 1 for every feature
    "scaled_mean": X_scaled.mean(), "scaled_std": X_scaled.std(),
}).round(3)                                                     # round everything to 3 decimals
print()
print(report.to_string())

rows: 270299 | columns: 15
NULLs in clustering features: 0


In [ ]:
report

,cap_low_1%,cap_high_99%,values_capped,skew_raw,skew_prepared,scaled_mean,scaled_std
loan_amount,1500.000,40000.000,2016,0.776,0.778,0.0,1.0
int_rate_pct,5.320,27.270,3496,0.783,0.692,-0.0,1.0
term_months,36.000,60.000,0,0.884,0.884,0.0,1.0
annual_inc,17000.000,276000.000,5337,47.192,0.068,-0.0,1.0
loan_to_income,0.025,0.732,5256,167.993,0.742,0.0,1.0
balance_to_income,0.056,7.272,5406,128.108,0.342,-0.0,1.0


In [ ]:
# Task 4, step 3: choose the number of clusters (k) for k-means.
# Fits k-means for k = 2 to 8 on the standardised features (X_scaled) and records two measures per k:
# inertia (falls as k grows; look for an 'elbow') and silhouette (higher = better separated clusters).
# Output: k_report, a 7-row table (one row per k).

import pandas as pd                                             # tables (DataFrames)
from sklearn.cluster import KMeans                              # the k-means clustering algorithm
from sklearn.metrics import silhouette_score                    # measures how well separated the clusters are

rows = []                                                       # will collect one result row per value of k
for k in range(2, 9):                                           # try k = 2, 3, ..., 8 clusters
    km = KMeans(n_clusters=k, n_init=5, random_state=42)        # k clusters; 5 random starts (best kept); fixed seed = repeatable
    labels = km.fit_predict(X_scaled)                           # fit on all customers; returns each customer's cluster number
    # silhouette on a random sample of 10,000 customers (all 270k is too slow); -1 to 1, higher = better separated
    sil = silhouette_score(X_scaled, labels, sample_size=10000, random_state=42)
    # store k, inertia (total distance to cluster centres; lower = tighter) and silhouette
    rows.append({"k": k, "inertia": round(km.inertia_), "silhouette": round(sil, 3)})
k_report = pd.DataFrame(rows)                                   # turn the list of results into a table
k_report

,k,inertia,silhouette
0,2,1212800,0.269
1,3,1044775,0.202
2,4,925754,0.207
3,5,836670,0.202
4,6,763876,0.199
5,7,717540,0.200
6,8,675593,0.198


In [ ]:
# Task 4, step 3 (chart): compare k = 2..8 visually.
# Left: inertia by k (look for an 'elbow' where the curve flattens). Right: silhouette by k (higher = better).
# Input: k_report from the previous cell.

import plotly.graph_objects as go                               # Plotly's building blocks for charts (traces)
from plotly.subplots import make_subplots                       # puts several charts side by side in one figure

# one figure with two charts side by side (1 row, 2 columns), each with its own title
fig = make_subplots(rows=1, cols=2, subplot_titles=("Inertia (lower = tighter clusters)",
                                                    "Silhouette (higher = better separated)"))
# left chart: inertia against k, drawn as a line with points
fig.add_trace(go.Scatter(x=k_report["k"], y=k_report["inertia"], mode="lines+markers"), row=1, col=1)
# right chart: silhouette against k, drawn as a line with points
fig.add_trace(go.Scatter(x=k_report["k"], y=k_report["silhouette"], mode="lines+markers"), row=1, col=2)
fig.update_xaxes(title="Number of clusters (k)", dtick=1)       # X-axis title on both charts; a tick at every whole k
# no legend (each chart has one line), 380 px tall, overall title
fig.update_layout(showlegend=False, height=380, title="Choosing the number of clusters")
fig

In [ ]:
# Task 4, step 4: fit the final k-means model (k = 5) and profile the five segments.
# Segments are numbered 1-5 by average loan amount (smallest to largest) so the numbering is stable.
# Default and late flags are NOT clustering inputs; they are used only to describe each segment afterwards.
# Output: profile, one row per segment (size, averages/medians, default rate, late rate).

import pandas as pd                                             # tables (DataFrames)
from sklearn.cluster import KMeans                              # the k-means clustering algorithm

km = KMeans(n_clusters=5, n_init=10, random_state=42)           # 5 clusters (chosen in the previous step); 10 random starts; fixed seed = repeatable
raw_labels = km.fit_predict(X_scaled)                           # fit on all customers; one cluster number (0-4, arbitrary order) per customer

seg = task4_customers.copy()                                    # copy of the customer table to add the cluster labels to
seg["is_default"] = seg["is_default"].astype(float)             # true/false -> 1.0/0.0, so an average becomes a rate
seg["is_late"] = seg["is_late"].astype(float)                   # same for the late flag
seg["raw_cluster"] = raw_labels                                 # attach each customer's cluster number

# cluster numbers ordered from smallest to largest average loan amount
order = seg.groupby("raw_cluster")["loan_amount"].mean().sort_values().index
# map each old cluster number to 1, 2, 3, 4, 5 in that order
relabel = {old: new for new, old in enumerate(order, start=1)}
seg["segment"] = seg["raw_cluster"].map(relabel)                # final segment number (1 = smallest average loan): stable, readable numbering

resolved = seg["status_group"] != "active"                      # loans with a known outcome: paid off or defaulted
active = seg["status_group"] == "active"                        # open loans: these can still be late

profile = seg.groupby("segment").agg(                           # one row per segment; each line below defines one column
    customers=("loan_id", "size"),                              # number of customers in the segment
    avg_loan_amount=("loan_amount", "mean"),                    # average loan amount
    avg_int_rate_pct=("int_rate_pct", "mean"),                  # average interest rate (%)
    # % of the segment's loans that are 60-month (True counts as 1, so the mean is a share)
    pct_60_month=("term_months", lambda s: 100 * (s == 60).mean()),
    median_annual_inc=("annual_inc", "median"),                 # median income (the middle value; barely moved by extreme incomes)
    median_loan_to_income=("loan_to_income", "median"),         # median loan amount divided by income
    # median total balance divided by income (leverage proxy)
    median_balance_to_income=("balance_to_income", "median"),
)
# insert a column at position 1: each segment's % of all customers
profile.insert(1, "share_pct", 100 * profile["customers"] / len(seg))
# default rate: % of RESOLVED loans that defaulted, per segment
profile["default_rate_resolved_pct"] = 100 * seg[resolved].groupby("segment")["is_default"].mean()
# late rate: % of ACTIVE loans that are late, per segment
profile["late_rate_active_pct"] = 100 * seg[active].groupby("segment")["is_late"].mean()
profile = profile.round(2)                                      # round every number to 2 decimals
profile

,customers,share_pct,avg_loan_amount,avg_int_rate_pct,pct_60_month,median_annual_inc,median_loan_to_income,median_balance_to_income,default_rate_resolved_pct,late_rate_active_pct
segment,,,,,,,,,,
1,67155,24.84,8010.5,12.03,0.83,60000.0,0.13,0.48,14.86,2.68
2,52812,19.54,9446.1,11.12,1.68,70000.0,0.13,3.12,10.61,2.48
3,39091,14.46,13829.44,14.14,3.61,37906.0,0.33,0.75,23.24,4.02
4,72961,26.99,20737.41,16.15,99.57,70100.0,0.28,1.74,32.30,3.87
5,38280,14.16,28099.05,10.69,13.13,120000.0,0.23,1.62,12.98,2.76


In [ ]:
# Task 4, step 5: bubble chart of the five customer segments.
# X = average loan amount, Y = default rate on resolved loans, bubble area = number of customers.
# Input: profile from the previous cell (one row per segment).

import plotly.express as px                                     # Plotly Express: high-level library for interactive charts

names = {                                                       # segment number -> descriptive name (chosen from each segment's profile)
    1: "Small loans, low leverage",
    2: "Small loans, high existing balances",
    3: "Low income, stretched",
    4: "60-month, higher-rate loans",
    5: "Large loans, high income",
}
plot_df = profile.reset_index()                                 # turn the 'segment' index into a normal column
# legend text such as "4 · 60-month, higher-rate loans" (number + name)
plot_df["label"] = plot_df["segment"].astype(str) + " · " + plot_df["segment"].map(names)

fig = px.scatter(                                               # one bubble per segment (5 bubbles)
    plot_df,                                                    # the segment profile table from the previous cell
    x="avg_loan_amount",                                        # X axis: average loan amount
    y="default_rate_resolved_pct",                              # Y axis: default rate on resolved loans (%)
    size="customers",                                           # bubble area = number of customers in the segment
    color="label",                                              # one colour per segment (also builds the legend)
    text="segment",                                             # segment number printed inside each bubble
    size_max=60,                                                # largest bubble is 60 pixels wide
    hover_name="label",                                         # bold title of the hover box
    hover_data={                                                # extra fields in the hover box, with number formats
        # formats: ':,' = thousands separator, ':.1f' / ':.2f' = 1 / 2 decimals
        "customers": ":,", "share_pct": ":.1f", "avg_int_rate_pct": ":.2f", "pct_60_month": ":.1f",
        # ':,.0f' = thousands separator, no decimals
        "median_annual_inc": ":,.0f", "late_rate_active_pct": ":.2f", "default_rate_resolved_pct": ":.1f",
        # False = hide (already visible on the axis, legend or inside the bubble)
        "avg_loan_amount": False, "label": False, "segment": False,
    },
    # friendly names shown in the hover box instead of column names
    labels={"customers": "Customers", "share_pct": "Share of customers (%)", "avg_int_rate_pct": "Avg interest rate (%)",
            "pct_60_month": "60-month loans (%)", "median_annual_inc": "Median income ($)",
            "late_rate_active_pct": "Late rate, active loans (%)", "default_rate_resolved_pct": "Default rate, resolved (%)",
            "label": "Segment"},
    # chart title
    title="Customer segments: loan size vs default rate (bubble size = number of customers)",
)
# bubble styling: number centred in white 15 pt text; slightly see-through bubbles with a thin white outline
fig.update_traces(textposition="middle center", textfont=dict(color="white", size=15),
                  marker=dict(opacity=0.85, line=dict(width=1, color="white")))
# X axis: title, $ prefix and thousands separators on ticks, fixed range 0 to 34,000
fig.update_xaxes(title="Average loan amount ($)", tickprefix="$", tickformat=",", range=[0, 34000])
# Y axis: title and fixed range 0 to 40 (%)
fig.update_yaxes(title="Default rate, resolved loans only (%)", range=[0, 40])
fig.update_layout(legend_title_text="Segment", height=520)      # legend heading; chart 520 px tall
fig

## Task 4 — Customer segmentation

**What was done:** customers were grouped with k-means clustering (k = 5) on six features: loan amount, interest rate, loan term, annual income, loan-to-income ratio, and total balance / income (a leverage proxy). Extreme values were capped at the 1st/99th percentile and skewed features log-transformed before standardising; no customers were dropped. Default and late status were **not** used to build the groups, only to describe them afterwards.

**Limits of the data:** there is no payment history and no credit limits, so "average delay" and "credit utilisation" could not be used. Each customer has exactly one loan, so "total loan amount" is simply that loan's amount.

**Why k = 5:** silhouette scores were about 0.20 for every k from 3 to 8, so they cannot pick a winner (k = 2 scores higher but mostly just separates 36- from 60-month loans). k = 5 was chosen because it was very stable across random restarts and, unlike k = 4, it isolates a group with high balances relative to income. The low silhouette means segments overlap; they are a useful description, not sharply separated natural groups.

| Segment | Customers | Avg loan | Avg rate | Median income | Default rate | Late rate |
|---|---:|---:|---:|---:|---:|---:|
| 1 · Small loans, low leverage | 67,155 (24.8%) | $8.0k | 12.0% | $60k | 14.9% | 2.7% |
| 2 · Small loans, high existing balances | 52,812 (19.5%) | $9.4k | 11.1% | $70k | 10.6% | 2.5% |
| 3 · Low income, stretched | 39,091 (14.5%) | $13.8k | 14.1% | $37.9k | 23.2% | 4.0% |
| 4 · 60-month, higher-rate loans | 72,961 (27.0%) | $20.7k | 16.2% | $70.1k | 32.3% | 3.9% |
| 5 · Large loans, high income | 38,280 (14.2%) | $28.1k | 10.7% | $120k | 13.0% | 2.8% |

Defaulted ÷ (paid off + defaulted). **Late / active loans.

**Findings**
- Although outcomes were not used to form the groups, default rates differ about threefold between segments (10.6% to 32.3%), which suggests the segments capture real differences in risk.
- **Segment 4** (99.6% 60-month loans, highest rates) has the highest default rate, but much of that is the known term effect. **Segment 3** (lowest income, highest loan-to-income) stands out among the mostly 36-month segments, with 23.2% default and the highest late rate (4.0%).
- **Segments 2 and 5** (higher incomes, lowest rates) have the lowest default rates.
- Late rates vary much less across segments (2.5%–4.0%) than default rates.

**Caveats:** default rates use resolved loans only and carry the snapshot-date caveat from Tasks 1–2, so they describe and compare segments rather than give precise risk levels. Segment membership shifts slightly (under 1% of customers) between runs.

# **TASK 5**

In [ ]:
# Task 5, step 1: which characteristics separate risky loans from safe ones?
# Cuts each characteristic into buckets (numeric ones into five equal-sized groups) and compares the default rate
# (resolved loans) and late rate (active loans) across buckets. Nothing is modelled or removed here.
# Outputs: drivers (one row per characteristic and bucket) and ranking (best-vs-worst bucket spread per characteristic).

import pandas as pd                                             # tables (DataFrames)

# sorted by loan_id so tied values always land in the same bucket, whatever order BigQuery returns rows in
d = task4_customers.sort_values("loan_id").reset_index(drop=True)
d["is_default"] = d["is_default"].astype(float)                 # true/false -> 1.0/0.0, so an average becomes a rate
d["is_late"] = d["is_late"].astype(float)                       # same for the late flag

# ---- buckets (computed on ALL loans so a bucket means the same thing for default and late) ----
QLAB = ["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (highest)"]        # labels for the five equal-sized groups


def quintile(s):                                                # cut a column into five equal-sized groups
    return pd.qcut(s.rank(method="first"), 5, labels=QLAB)      # rank first (ties split evenly), then cut into 5


# 36 -> "36 months" (text label)
d["term"] = d["term_months"].astype(int).astype(str) + " months"
# keep MORTGAGE / RENT / OWN; group the tiny remaining categories as OTHER
d["home"] = d["home_ownership"].where(d["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"]), "OTHER")
d["interest rate"] = quintile(d["int_rate_pct"])                # interest-rate fifth (Q1 = lowest rates)
d["loan-to-income"] = quintile(d["loan_to_income"])             # loan-to-income fifth
d["annual income"] = quintile(d["annual_inc"])                  # income fifth
d["balance-to-income"] = quintile(d["balance_to_income"])       # balance-to-income fifth
d["loan amount"] = quintile(d["loan_amount"])                   # loan-amount fifth

CHARACTERISTICS = {                                             # characteristics to analyse: display label -> column in d
    "grade": "grade", "term": "term", "interest rate": "interest rate", "loan-to-income": "loan-to-income",
    "annual income": "annual income", "balance-to-income": "balance-to-income", "loan amount": "loan amount",
    "home ownership": "home", "application type": "application_type",
}

resolved = d[d["status_group"] != "active"]                     # loans with a known outcome: paid off or defaulted
active = d[d["status_group"] == "active"]                       # open loans: these can still be late
overall_default = 100 * resolved["is_default"].mean()           # overall default rate (%) on resolved loans
overall_late = 100 * active["is_late"].mean()                   # overall late rate (%) on active loans

parts = []                                                      # one summary table per characteristic, combined at the end
for label, col in CHARACTERISTICS.items():                      # repeat for each characteristic
    # per bucket: number of resolved loans and their default rate (%)
    r = resolved.groupby(col, observed=True)["is_default"].agg(resolved_loans="size", default_rate_pct=lambda s: 100 * s.mean())
    # per bucket: number of active loans and their late rate (%)
    a = active.groupby(col, observed=True)["is_late"].agg(active_loans="size", late_rate_pct=lambda s: 100 * s.mean())
    # put the two side by side; the bucket becomes a normal column named 'bucket'
    t = r.join(a, how="outer").reset_index().rename(columns={col: "bucket"})
    t.insert(0, "characteristic", label)                        # add a first column naming the characteristic
    parts.append(t)                                             # store this characteristic's table
drivers = pd.concat(parts, ignore_index=True)                   # stack all the tables into one
drivers["bucket"] = drivers["bucket"].astype(str)               # bucket labels as plain text
# default rate relative to the overall rate (2.0 = twice the average)
drivers["default_lift"] = drivers["default_rate_pct"] / overall_default
drivers = drivers.round(2)                                      # round to 2 decimals

# ---- ranking: how far apart are the best and worst bucket of each characteristic? ----
# Buckets with fewer than MIN_N loans are ignored here (a 62-loan bucket can show any rate by chance).
MIN_N = 500                                                     # smallest bucket size we trust
# flag buckets that are too small to trust (true/false)
drivers["small_bucket"] = (drivers["resolved_loans"] < MIN_N) | (drivers["active_loans"] < MIN_N)

# default rates of the large-enough buckets, grouped by characteristic
big_r = drivers[drivers["resolved_loans"] >= MIN_N].groupby("characteristic")["default_rate_pct"]
# late rates of the large-enough buckets, grouped by characteristic
big_a = drivers[drivers["active_loans"] >= MIN_N].groupby("characteristic")["late_rate_pct"]
ranking = pd.DataFrame({                                        # one row per characteristic
    # lowest and highest default rate across a characteristic's buckets
    "default_lowest": big_r.min(), "default_highest": big_r.max(),
    "late_lowest": big_a.min(), "late_highest": big_a.max(),    # lowest and highest late rate
})
# spread = highest minus lowest bucket rate, in percentage points: how strongly it separates risk
ranking["default_spread_pts"] = ranking["default_highest"] - ranking["default_lowest"]
# the same spread for the late rate
ranking["late_spread_pts"] = ranking["late_highest"] - ranking["late_lowest"]
# strongest separators first; round to 1 decimal
ranking = ranking.sort_values("default_spread_pts", ascending=False).round(1)

# print the overall rates for reference
print(f"overall default rate (resolved): {overall_default:.2f}% | overall late rate (active): {overall_late:.2f}%")
ranking

overall default rate (resolved): 18.96% | overall late rate (active): 3.19%


,default_lowest,default_highest,late_lowest,late_highest,default_spread_pts,late_spread_pts
characteristic,,,,,,
grade,5.8,48.1,1.0,14.6,42.3,13.6
interest rate,6.0,33.8,1.0,6.3,27.8,5.4
loan-to-income,12.0,28.5,2.7,3.9,16.5,1.2
term,15.0,31.4,3.0,3.6,16.4,0.7
annual income,14.5,22.6,3.0,3.5,8.1,0.5
loan amount,14.2,22.3,2.7,3.6,8.1,0.9
balance-to-income,15.9,23.1,2.7,3.6,7.3,0.8
home ownership,16.3,22.3,2.9,3.5,6.0,0.6
application type,17.8,19.0,3.2,3.2,1.1,0.0


In [ ]:
drivers

,characteristic,bucket,resolved_loans,default_rate_pct,active_loans,late_rate_pct,default_lift,small_bucket
0,grade,A,16671,5.81,41200,0.98,0.31,False
1,grade,B,26897,12.27,52175,2.29,0.65,False
2,grade,C,26580,20.94,49260,3.70,1.10,False
3,grade,D,14494,28.48,24382,5.51,1.50,False
4,grade,E,6666,38.12,6763,8.34,2.01,False
5,grade,F,2245,46.15,1760,11.76,2.43,False
6,grade,G,671,48.14,535,14.58,2.54,False
7,term,36 months,71590,15.02,118182,2.97,0.79,False
8,term,60 months,22634,31.43,57893,3.63,1.66,False
9,interest rate,Q1 (lowest),16836,6.02,37224,0.96,0.32,False


In [ ]:
# Task 5, step 2: which characteristics predict default and late payment, taken together?
# Two logistic-regression models, each scored on a held-out 30% test set:
# default model: resolved loans only (paid off vs defaulted); late model: active loans only (current vs late).
# Grade is left out of the models because the interest rate is derived from it; its stand-alone strength is still shown.
# Needs task4_customers (SQL cell) and X (Task 4 preparation cell). Outputs: model_summary, default_table, late_table.

import numpy as np                                              # numerical functions (exp, maximum, where)
import pandas as pd                                             # tables (DataFrames)
from sklearn.linear_model import LogisticRegression             # the model: predicts a yes/no outcome from several features at once
from sklearn.metrics import roc_auc_score                       # AUC: how well scores rank risky loans above safe ones (0.5 = coin flip, 1 = perfect)
from sklearn.model_selection import train_test_split            # splits the data into a training part and a held-out test part
from sklearn.preprocessing import StandardScaler                # rescales each feature to mean 0, std 1

base = task4_customers.copy()                                   # the customer table (raw columns and outcomes)
base["is_default"] = base["is_default"].astype(int)             # true/false -> 1/0 (the target the model predicts)
base["is_late"] = base["is_late"].astype(int)                   # same for the late flag

feat = X.copy()                                                 # start from X: capped and log-transformed, not yet standardised (Task 4 preparation cell)
feat["term_60"] = (base["term_months"] == 60).astype(float)     # 1.0 for a 60-month loan, 0.0 for a 36-month loan
feat = feat.drop(columns=["term_months"])                       # remove the raw term column (replaced by term_60)
# 1.0 if the borrower rents (MORTGAGE is the baseline)
feat["home_RENT"] = (base["home_ownership"] == "RENT").astype(float)
# 1.0 if the borrower owns outright
feat["home_OWN"] = (base["home_ownership"] == "OWN").astype(float)
# 1.0 for a joint application
feat["joint_application"] = (base["application_type"] == "Joint").astype(float)
# grade as a number, A=1 ... G=7 (only for the stand-alone comparison; not a model input)
grade_num = base["grade"].map({g: i for i, g in enumerate("ABCDEFG", start=1)})


def fit_and_report(mask, target):                               # fit one model on the loans selected by mask, predicting the target
    Xm, y = feat[mask], base.loc[mask, target]                  # inputs and target for the selected loans only
    # 70% training / 30% held-out test; stratify keeps the same event share in both; fixed seed
    X_tr, X_te, y_tr, y_te = train_test_split(Xm, y, test_size=0.3, random_state=42, stratify=y)
    scaler = StandardScaler().fit(X_tr)                         # learn each feature's mean and std from the training data only
    # fit on the scaled training data (max_iter=1000 = enough steps to converge)
    model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)
    # AUC on the held-out test data: predicted probability of the event vs what actually happened
    test_auc = roc_auc_score(y_te, model.predict_proba(scaler.transform(X_te))[:, 1])

    # AUC of each feature used ALONE as a risk score (0.5 = no signal)
    alone = {c: roc_auc_score(y_te, X_te[c]) for c in X_te.columns}
    # the same for grade, on the same test loans
    alone["grade (A=1 ... G=7)"] = roc_auc_score(y_te, grade_num.loc[X_te.index])
    table = pd.DataFrame({"auc_alone": pd.Series(alone)})       # table with one row per feature
    # odds ratio: how much a 1-standard-deviation rise multiplies the odds of the event, other features held fixed (above 1 = riskier)
    table["odds_ratio_per_1sd"] = pd.Series(np.exp(model.coef_[0]), index=X_te.columns)
    # strength ignoring direction: an AUC of 0.45 is as informative as 0.55, just reversed
    table["auc_alone_strength"] = np.maximum(table["auc_alone"], 1 - table["auc_alone"])
    # direction: does a higher value of the feature go with more risk?
    table["higher_value_means"] = np.where(table["auc_alone"] >= 0.5, "riskier", "safer")
    # strongest features first; round to 3 decimals
    table = table.sort_values("auc_alone_strength", ascending=False).round(3)
    # one-row summary: target, loans modelled, event rate (%), test rows, and the model's test AUC
    summary = {"target": target, "loans_modelled": len(Xm), "event_rate_pct": round(100 * y.mean(), 2),
               "test_rows": len(y_te), "model_test_auc": round(test_auc, 3)}
    return summary, table                                       # hand both results back to the caller


# default model: resolved loans only, predicting default
s1, default_table = fit_and_report(base["status_group"] != "active", "is_default")
# late model: active loans only, predicting late payment
s2, late_table = fit_and_report(base["status_group"] == "active", "is_late")
model_summary = pd.DataFrame([s1, s2])                          # combine the two summaries into one table
model_summary

,target,loans_modelled,event_rate_pct,test_rows,model_test_auc
0,is_default,94224,18.96,28268,0.697
1,is_late,176075,3.19,52823,0.674


In [ ]:
# Task 5, step 3: how would candidate lending rules have performed on historical loans?
# Each rule flags loans using only information known at application time (grade, rate, term, loan-to-income).
# For each rule: default rate inside vs outside it, share of all defaults it catches, share of good loans it would
# also flag (its cost), and the late rate on active loans inside vs outside. Output: rules_table (one row per rule).

import pandas as pd                                             # tables (DataFrames)

d = task4_customers.copy()                                      # SQL result from Hex; a copy keeps the original unchanged
d["is_default"] = d["is_default"].astype(bool)                  # make sure the flag is true/false (needed for ~ and sums below)
d["is_late"] = d["is_late"].astype(bool)                        # same for the late flag

rules = {                                                       # each rule is a true/false test per loan: True = the loan is flagged
    "Grade D-G": d["grade"].isin(list("DEFG")),                 # grade D, E, F or G
    "Grade E-G": d["grade"].isin(list("EFG")),                  # grade E, F or G
    "Rate >= 20%": d["int_rate_pct"] >= 20,                     # interest rate of 20% or more
    "Rate >= 25%": d["int_rate_pct"] >= 25,                     # interest rate of 25% or more
    # 60-month term AND rate of 16% or more (& = both conditions must hold)
    "60-month AND rate >= 16%": (d["term_months"] == 60) & (d["int_rate_pct"] >= 16),
    "Loan-to-income >= 0.35": d["loan_to_income"] >= 0.35,      # loan amount is at least 35% of annual income
    # 60-month term AND loan-to-income of 0.30 or more
    "60-month AND loan-to-income >= 0.30": (d["term_months"] == 60) & (d["loan_to_income"] >= 0.30),
    # either condition is enough (| = or): grade D-G, or 60-month with loan-to-income of 0.30 or more
    "Grade D-G OR (60-month AND LTI >= 0.30)": d["grade"].isin(list("DEFG")) | ((d["term_months"] == 60) & (d["loan_to_income"] >= 0.30)),
}

resolved = d["status_group"] != "active"                        # loans with a known outcome: paid off or defaulted
active = ~resolved                                              # open loans: these can still be late (~ = not)
total_defaults = d.loc[resolved, "is_default"].sum()            # all defaults among resolved loans (True counts as 1)
total_good = (~d.loc[resolved, "is_default"]).sum()             # all paid-off loans (resolved loans that did not default)

rows = []                                                       # will collect one result row per rule
for name, flag in rules.items():                                # for each rule: its name and its true/false test
    fr, nr = d[resolved & flag], d[resolved & ~flag]            # resolved loans flagged (fr) and not flagged (nr)
    fa, na = d[active & flag], d[active & ~flag]                # active loans flagged (fa) and not flagged (na)
    rows.append({                                               # one row of results for this rule
        "rule": name,                                           # rule name
        # % of resolved loans the rule flags
        "flagged_pct_of_resolved": 100 * len(fr) / resolved.sum(),
        # default rate (%) among flagged resolved loans
        "default_rate_flagged": 100 * fr["is_default"].mean(),
        # default rate (%) among resolved loans NOT flagged
        "default_rate_not_flagged": 100 * nr["is_default"].mean(),
        # % of ALL defaults that the rule flags
        "defaults_caught_pct": 100 * fr["is_default"].sum() / total_defaults,
        # % of paid-off loans the rule would also flag (the cost of the rule)
        "good_loans_flagged_pct": 100 * (~fr["is_default"]).sum() / total_good,
        "late_rate_flagged": 100 * fa["is_late"].mean(),        # late rate (%) among flagged active loans
        "late_rate_not_flagged": 100 * na["is_late"].mean(),    # late rate (%) among active loans NOT flagged
    })
rules_table = pd.DataFrame(rows).round({                        # turn the rows into a table; round each column
    # decimals per column: 1 for most, 2 for the late rates
    "flagged_pct_of_resolved": 1, "default_rate_flagged": 1, "default_rate_not_flagged": 1,
    "defaults_caught_pct": 1, "good_loans_flagged_pct": 1, "late_rate_flagged": 2, "late_rate_not_flagged": 2,
})
# overall rates for reference
print(f"overall default rate (resolved): {100 * d.loc[resolved, 'is_default'].mean():.1f}% | "
      f"overall late rate (active): {100 * d.loc[active, 'is_late'].mean():.2f}%")
rules_table

overall default rate (resolved): 19.0% | overall late rate (active): 3.19%


,rule,flagged_pct_of_resolved,default_rate_flagged,default_rate_not_flagged,defaults_caught_pct,good_loans_flagged_pct,late_rate_flagged,late_rate_not_flagged
0,Grade D-G,25.6,33.3,14.0,44.9,21.0,6.56,2.40
1,Grade E-G,10.2,40.7,16.5,21.8,7.4,9.37,2.85
2,Rate >= 20%,8.5,37.0,17.3,16.5,6.6,7.47,2.75
3,Rate >= 25%,2.1,40.4,18.5,4.5,1.6,9.17,3.03
4,60-month AND rate >= 16%,12.4,39.0,16.1,25.5,9.3,5.89,2.81
5,Loan-to-income >= 0.35,13.9,29.2,17.3,21.4,12.2,3.85,3.05
6,60-month AND loan-to-income >= 0.30,10.1,37.1,16.9,19.8,7.8,3.95,3.07
7,Grade D-G OR (60-month AND LTI >= 0.30),29.6,32.5,13.3,50.8,24.7,5.24,2.40


In [ ]:
default_table

,auc_alone,odds_ratio_per_1sd,auc_alone_strength,higher_value_means
grade (A=1 ... G=7),0.682,NaN,0.682,riskier
int_rate_pct,0.681,1.634,0.681,riskier
loan_to_income,0.604,1.230,0.604,riskier
term_60,0.594,1.244,0.594,riskier
annual_inc,0.446,0.965,0.554,safer
loan_amount,0.551,0.939,0.551,riskier
home_RENT,0.545,1.135,0.545,riskier
balance_to_income,0.465,0.935,0.535,safer
joint_application,0.498,0.944,0.502,safer
home_OWN,0.500,1.036,0.500,riskier


In [ ]:
late_table

,auc_alone,odds_ratio_per_1sd,auc_alone_strength,higher_value_means
grade (A=1 ... G=7),0.672,NaN,0.672,riskier
int_rate_pct,0.668,1.756,0.668,riskier
loan_to_income,0.548,1.069,0.548,riskier
loan_amount,0.538,1.056,0.538,riskier
term_60,0.529,0.874,0.529,riskier
home_RENT,0.519,1.058,0.519,riskier
annual_inc,0.491,1.038,0.509,safer
balance_to_income,0.491,0.999,0.509,safer
joint_application,0.503,0.956,0.503,riskier
home_OWN,0.500,1.039,0.500,safer


## Task 5 — Insights & recommendations

**Headline:** the lender's own risk grade and interest rate are, by far, the strongest predictors of default and late payment. Everything else in the data adds only a little. Simple rules based on grade, term and loan-to-income can flag high-risk loans, but every rule also flags many loans that turn out fine, so they suit review, pricing and limit decisions rather than automatic declines.

### What customer or loan characteristics best predict default or late payment?

1. **Grade / interest rate (strongest).** Default rate among resolved loans rises from **5.8% (grade A) to 48.1% (grade G)**; by interest-rate fifth it rises from **6.1% to 33.7%**. Late rate among active loans rises from 1.0% (A) to 14.6% (G). Grade or rate alone scores an AUC of about 0.68 for default and 0.67 for late payment; a model using all available characteristics reaches only **0.697 (default)** and **0.674 (late)**, so the rest of the data adds about 0.02.
2. **Loan term.** 60-month loans default at **31.4% vs 15.0%** for 36-month loans, and the effect remains when other factors are held fixed (odds ratio 1.24 per standard deviation).
3. **Loan-to-income.** Default rises from **12.0% (lowest fifth) to 28.5% (highest fifth)** (odds ratio 1.23). It does not help predict *late* payment (see below).
4. **Weaker signals.** Lower income (22.7% default in the lowest income fifth vs 14.5% in the highest) and renting (22.3% vs 16.3% for mortgage holders).
5. **No useful signal.** Joint vs individual application (17.8% vs 19.0%), and loan amount and balance-to-income once other factors are considered.
6. **Late payment is harder to predict than default.** Only interest rate/grade separates late loans meaningfully; the model's AUC is 0.674, and other characteristics change late rates by less than about 1.5 points.

Predictive strength is moderate overall (AUC around 0.70, where 0.5 is a coin flip). The fact that the lender's own pricing carries most of the signal suggests existing underwriting already captures much of what this data can tell us.

### If designing lending rules, what would I flag or change?

Evaluated on historical loans (share of resolved loans flagged; default rate flagged vs not flagged; share of all defaults caught; share of good loans also flagged):

| Rule | Flagged | Default: flagged vs not | Defaults caught | Good loans flagged |
|---|---:|---:|---:|---:|
| Grade E–G | 10.2% | 40.7% vs 16.5% | 21.8% | 7.4% |
| 60-month AND rate ≥ 16% | 12.4% | 39.0% vs 16.1% | 25.5% | 9.3% |
| Loan-to-income ≥ 0.35 | 13.9% | 29.2% vs 17.3% | 21.4% | 12.2% |
| Grade D–G | 25.6% | 33.3% vs 14.0% | 44.9% | 21.0% |

1. **Flag grade E–G and 60-month loans priced at 16% or more** for tighter limits, higher pricing or manual review. They are about 10–12% of loans and default at about 39–41%, against roughly 16% for the rest.
2. **Treat high loan-to-income (≥ 0.35) as a secondary flag** for default risk, for example by capping loan size or requiring income verification. It adds little for late payment (3.85% vs 3.05%).
3. **Add early-warning monitoring for high-rate loans:** loans at 20%+ are late at 7.5% vs 2.8%.
4. **Do not add rules for joint applications or loan amount alone;** the data shows no useful signal.
5. **Use rules as triggers, not automatic declines:** even the best rules are wrong for 60–67% of the loans they flag (those loans did not default).
6. **Fix the data, which is the biggest change:** record payment dates and days late, credit limits/utilisation and outcome dates, so lateness and default can be measured directly and rules can be tested properly.

### Limitations
- Default rates use resolved loans only, and loan statuses appear not to share one snapshot date (see Tasks 1–2), so rates are for comparing groups, not exact risk levels.
- There is no payment history: "late" means late *at the time of the snapshot*.
- The data contains only approved loans, so we cannot tell how rejected applicants would have behaved.
- Rule thresholds (16%, 0.30, 0.35) were chosen by eye from the buckets and tested on the same data; they need validating on new data.
- These are associations, not causes: the interest rate reflects the lender's own risk view, so it predicts default without necessarily causing it.

Where the claims come from

- Grade and interest-rate bucket figures, term, loan-to-income, income, home ownership and joint vs individual: step 1 (drivers).
- AUC figures: your Hex tables (model_summary, default_table, late_table).
- Rule figures: step 3 (rules_table), plus the 7.5% vs 2.8% for rate ≥ 20%.
- "60–67% of flagged loans did not default": the flagged default rates are 33–41%, so 59–67% of flagged loans did not default. It's rounded in the text.